In [ ]:
import os
import os.path as op
import mne
import nibabel as nib
import numpy as np

In [ ]:
# Setup paths
path = '/Users/immlab/Desktop/IMM-Lab'
meg_path = op.join(path, 'MEG')
mri_path = op.join(path, 'MRI')

# Load source space
src_path = op.join(mri_path, 'fsaverage', 'bem', 'fsaverage-mixed-src.fif')
src = mne.read_source_spaces(src_path, verbose=False)

In [ ]:
def process_and_convert_stc_to_nifti(input_folder, output_folder, src, resample_freq=None):
    """
    Convert all STC files in a folder to NIfTI format.
    For each STC, finds peak activation time, extracts ±20ms around peak (41 time points),
    and saves two NIfTI files: one with all 41 volumes, one with the average volume.
    
    Parameters:
    - input_folder: Path to folder containing STC files
    - output_folder: Path to save NIfTI files (will be created if doesn't exist)
    - src: MNE source space object
    - resample_freq: Optional frequency to resample to (e.g., 100, 200, 500)
    """
    
    os.makedirs(output_folder, exist_ok=True)
    
    stc_files = [f for f in os.listdir(input_folder) if f.endswith('-stc.h5')]
    
    if not stc_files:
        print(f"No STC files found in {input_folder}")
        return
    
    print(f"Found {len(stc_files)} STC files in {input_folder}")
    print(f"Output folder: {output_folder}")
    
    total_4d_size = 0
    total_3d_size = 0
    
    for stc_file in stc_files:
        try:
            stc_path = op.join(input_folder, stc_file)
            stc = mne.read_source_estimate(stc_path, verbose=False)
            
            print(f"\nProcessing {stc_file}:")
            print(f"  Original: {1/stc.tstep:.0f} Hz, {len(stc.times)} time points, {stc.times[0]:.3f} to {stc.times[-1]:.3f} s")
            
            if resample_freq is not None:
                print(f"  Resampling from {1/stc.tstep:.0f} Hz to {resample_freq} Hz")
                stc = stc.copy().resample(resample_freq)
            
            global_activation = np.sum(np.abs(stc.data), axis=0)
            peak_time_idx = np.argmax(global_activation)
            peak_time = stc.times[peak_time_idx]
            peak_value = global_activation[peak_time_idx]
            
            print(f"  Peak activation: {peak_value:.2e} at t = {peak_time:.3f} s (index {peak_time_idx})")
            
            time_window_ms = 20  # milliseconds
            time_window_s = time_window_ms / 1000  # convert to seconds
            
            samples_per_20ms = int(time_window_s / stc.tstep)
            
            start_idx = max(0, peak_time_idx - samples_per_20ms)
            end_idx = min(len(stc.times), peak_time_idx + samples_per_20ms + 1)
            
            actual_window = end_idx - start_idx
            if actual_window < 41 and len(stc.times) >= 41:
                deficit = 41 - actual_window
                expand_left = deficit // 2
                expand_right = deficit - expand_left
                
                new_start = max(0, start_idx - expand_left)
                new_end = min(len(stc.times), end_idx + expand_right)
                
                start_idx, end_idx = new_start, new_end
            
            stc_windowed = stc.copy().crop(stc.times[start_idx], stc.times[end_idx-1])
            
            print(f"  Extracted window: {len(stc_windowed.times)} time points from {stc_windowed.times[0]:.3f} to {stc_windowed.times[-1]:.3f} s")
            print(f"  Window duration: {(stc_windowed.times[-1] - stc_windowed.times[0])*1000:.1f} ms")
            
            vol_stc = stc_windowed.volume()
            
            nii_4d = vol_stc.as_volume(src, mri_resolution=True)
            
            nii_data_avg = np.mean(nii_4d.get_fdata(), axis=-1)
            
            nii_3d = nib.Nifti1Image(nii_data_avg, nii_4d.affine, nii_4d.header)
            
            base_name = stc_file.replace('-stc.h5', '')
            suffix = f"_resampled_{resample_freq}Hz" if resample_freq else ""
            
            nii_4d_filename = f"{base_name}{suffix}_peak_window_4D.nii.gz"
            nii_4d_path = op.join(output_folder, nii_4d_filename)
            
            nii_3d_filename = f"{base_name}{suffix}_peak_window_avg.nii.gz"
            nii_3d_path = op.join(output_folder, nii_3d_filename)
            
            nib.save(nii_4d, nii_4d_path)
            nib.save(nii_3d, nii_3d_path)
            
        except Exception as e:
            print(f"Error processing {stc_file}: {str(e)}")
    
    print(f"Files saved to: {output_folder}")

In [ ]:
adult_input_folder = op.join(meg_path, 'adult_morphed_averaged')
adult_output_folder = op.join(meg_path, 'adult_morphed_averaged', 'nifti_files')

process_and_convert_stc_to_nifti(adult_input_folder, adult_output_folder, src)

In [ ]:
child_input_folder = op.join(meg_path, 'child_morphed_averaged')
child_output_folder = op.join(meg_path, 'child_morphed_averaged', 'nifti_files')

process_and_convert_stc_to_nifti(child_input_folder, child_output_folder, src)

In [ ]:
# MEMORY USAGE ESTIMATION AND DIAGNOSTICS

def estimate_nifti_memory_usage(stc_folder, src, resample_freq=None):
    """
    Estimate memory usage and file sizes for NIfTI conversion.
    """
    print("=== MEMORY USAGE ESTIMATION ===")
    
    # Find first STC file for estimation
    stc_files = [f for f in os.listdir(stc_folder) if f.endswith('-stc.h5')]
    if not stc_files:
        print("No STC files found for estimation")
        return
    
    # Load one STC file for estimation
    stc_path = op.join(stc_folder, stc_files[0])
    stc = mne.read_source_estimate(stc_path, verbose=False)
    
    print(f"Using {stc_files[0]} for estimation:")
    print(f"  Original sampling: {1/stc.tstep:.0f} Hz")
    print(f"  Original time points: {len(stc.times)}")
    print(f"  Original duration: {stc.times[-1] - stc.times[0]:.3f} s")
    print(f"  Number of vertices: {stc.data.shape[0]}")
    
    # Resample if specified
    if resample_freq is not None:
        stc_resampled = stc.copy().resample(resample_freq)
        print(f"  After resampling to {resample_freq} Hz: {len(stc_resampled.times)} time points")
        stc = stc_resampled
    
    # Calculate ±20ms window
    time_window_ms = 20
    samples_per_20ms = int((time_window_ms / 1000) / stc.tstep)
    
    # Window will be approximately 41 time points
    window_points = min(41, 2 * samples_per_20ms + 1)
    print(f"  Window around peak: {window_points} time points ({window_points * stc.tstep * 1000:.1f} ms)")
    
    # Convert to volume to get NIfTI dimensions
    vol_stc = stc.volume()
    nii_temp = vol_stc.as_volume(src, mri_resolution=True)
    
    print(f"\nNIfTI properties:")
    print(f"  3D volume shape: {nii_temp.shape[:3]}")
    print(f"  Voxel dimensions: {nii_temp.header.get_zooms()[:3]} mm")
    print(f"  Data type: {nii_temp.get_fdata().dtype}")
    
    # Calculate file sizes
    voxels_per_volume = np.prod(nii_temp.shape[:3])
    bytes_per_voxel = 8 if nii_temp.get_fdata().dtype == np.float64 else 4
    
    # 4D NIfTI (41 volumes)
    size_4d_mb = (voxels_per_volume * window_points * bytes_per_voxel) / (1024**2)
    
    # 3D NIfTI (averaged volume)
    size_3d_mb = (voxels_per_volume * bytes_per_voxel) / (1024**2)
    
    print(f"\nEstimated file sizes per STC:")
    print(f"  4D NIfTI ({window_points} volumes): {size_4d_mb:.1f} MB")
    print(f"  3D NIfTI (averaged): {size_3d_mb:.1f} MB")
    print(f"  Total per STC: {size_4d_mb + size_3d_mb:.1f} MB")
    
    print(f"\nEstimated total for all {len(stc_files)} STC files:")
    print(f"  Total 4D files: {size_4d_mb * len(stc_files):.1f} MB")
    print(f"  Total 3D files: {size_3d_mb * len(stc_files):.1f} MB")
    print(f"  Grand total: {(size_4d_mb + size_3d_mb) * len(stc_files):.1f} MB")
    
    return {
        'size_4d_mb': size_4d_mb,
        'size_3d_mb': size_3d_mb,
        'total_files': len(stc_files),
        'window_points': window_points
    }

# Run diagnostics for adult data
print("ADULT DATA DIAGNOSTICS:")
adult_stats = estimate_nifti_memory_usage(adult_input_folder, src, resample_freq=500)

print("\n" + "="*50)
print("CHILD DATA DIAGNOSTICS:")
child_stats = estimate_nifti_memory_usage(child_input_folder, src, resample_freq=500)

In [ ]:
# RESAMPLING COMPARISON - Impact on file size

def compare_resampling_options(stc_folder, src):
    """
    Compare file sizes with different resampling frequencies.
    """
    print("=== RESAMPLING COMPARISON ===")
    
    # Test different resampling frequencies
    frequencies = [None, 1000, 500, 200, 100]  # None = original
    
    # Find first STC file
    stc_files = [f for f in os.listdir(stc_folder) if f.endswith('-stc.h5')]
    if not stc_files:
        return
    
    stc_path = op.join(stc_folder, stc_files[0])
    stc_orig = mne.read_source_estimate(stc_path, verbose=False)
    
    print(f"Original STC: {1/stc_orig.tstep:.0f} Hz, {len(stc_orig.times)} time points")
    print(f"Analyzing {len(stc_files)} STC files\n")
    
    results = []
    
    for freq in frequencies:
        stc = stc_orig.copy()
        
        if freq is not None:
            if freq >= 1/stc.tstep:
                print(f"Skipping {freq} Hz (higher than original {1/stc.tstep:.0f} Hz)")
                continue
            stc = stc.resample(freq)
        
        # Calculate ±20ms window
        time_window_ms = 20
        samples_per_20ms = int((time_window_ms / 1000) / stc.tstep)
        window_points = min(41, 2 * samples_per_20ms + 1)
        
        # Estimate file size
        vol_stc = stc.volume()
        nii_temp = vol_stc.as_volume(src, mri_resolution=True)
        voxels_per_volume = np.prod(nii_temp.shape[:3])
        bytes_per_voxel = 8 if nii_temp.get_fdata().dtype == np.float64 else 4
        
        size_4d_mb = (voxels_per_volume * window_points * bytes_per_voxel) / (1024**2)
        size_3d_mb = (voxels_per_volume * bytes_per_voxel) / (1024**2)
        total_per_stc = size_4d_mb + size_3d_mb
        total_all_files = total_per_stc * len(stc_files)
        
        freq_label = f"{freq} Hz" if freq else f"{1/stc_orig.tstep:.0f} Hz (original)"
        
        results.append({
            'frequency': freq_label,
            'window_points': window_points,
            'size_per_stc_mb': total_per_stc,
            'total_all_mb': total_all_files
        })
        
        print(f"{freq_label:>15}: {window_points:2d} points, {total_per_stc:5.1f} MB per STC, {total_all_files:6.1f} MB total")
    
    # Calculate reduction ratios
    if len(results) > 1:
        original_size = results[0]['total_all_mb']
        print(f"\nFile size reductions compared to original:")
        for r in results[1:]:
            reduction = (1 - r['total_all_mb'] / original_size) * 100
            print(f"{r['frequency']:>15}: {reduction:5.1f}% smaller")
    
    return results

# Compare different resampling options
print("ADULT DATA - Resampling comparison:")
adult_comparison = compare_resampling_options(adult_input_folder, src)

print("\n" + "="*60)
print("CHILD DATA - Resampling comparison:")
child_comparison = compare_resampling_options(child_input_folder, src)